In [3]:
# 连接数据库

import sqlite3

db_path = "db/bible.db"

conn = sqlite3.connect(db_path)
cursor = conn.cursor()

print("数据库连接成功")

数据库连接成功


In [8]:
# 快速备份

import sqlite3

src = "db/bible.db"
backup = "db/bible_backup.sqlite"

with sqlite3.connect(src) as src_conn:
    with sqlite3.connect(backup) as bak_conn:
        src_conn.backup(bak_conn)

print("SQLite 备份完成")

SQLite 备份完成


In [5]:
# 快速恢复

import sqlite3

backup = "db/bible_backup.sqlite"
target = "db/bible.db"

with sqlite3.connect(backup) as src_conn:
    with sqlite3.connect(target) as dst_conn:
        src_conn.backup(dst_conn)

print("数据库恢复完成")

数据库恢复完成


In [5]:
# 填充指定章（手动）的 tokens 表，从 verse 表获取数据

import sqlite3
from typing import List, Tuple

DB_PATH = "db/bible.db"


def segment_english_preserve(text: str) -> List[Tuple[str, str]]:
    if not text:
        return []

    tokens = []
    current = ""
    current_type = None

    def flush():
        nonlocal current, current_type
        if current:
            tokens.append((current, current_type))
            current = ""
            current_type = None

    for ch in text:
        if ch.isalpha():
            t = "word"
        elif ch.isdigit():
            t = "number"
        elif ch.isspace():
            t = "space"
        else:
            t = "punct"

        if t != current_type:
            flush()
            current_type = t

        current += ch

    flush()
    return tokens


def build_tokens_for_chapter(book_abbr: str, book_id: int, chapter: int):
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()

    print(f"📖 处理章节：{book_abbr} {chapter}")

    cursor.execute("""
        SELECT v.id, v.verse, v.text_en
        FROM verses v
        WHERE v.book_id = ?
          AND v.chapter = ?
          AND v.text_en IS NOT NULL
        ORDER BY v.verse
    """, (book_id, chapter))

    verses = cursor.fetchall()
    print(f"  需要处理的 verse 数：{len(verses)}")

    token_rows = []

    for verse_id, verse_num, text_en in verses:
        tokens = segment_english_preserve(text_en)

        reconstructed = "".join(t for t, _ in tokens)
        if reconstructed != text_en:
            print("❌ 拼接不一致")
            print("verse:", verse_id)
            print("原句:", repr(text_en))
            print("拼接:", repr(reconstructed))
            continue

        for idx, (token, token_type) in enumerate(tokens, start=1):
            token_id = f"{book_abbr}.{chapter}.{verse_num}.{idx}"

            token_rows.append((
                token_id,
                book_id,
                chapter,
                verse_num,
                idx,
                token,
                token_type,
                None
            ))

    cursor.executemany("""
        INSERT INTO tokens (
            id,
            book_id,
            chapter,
            verse,
            token_id,
            token,
            type,
            entity_key
        ) VALUES (?, ?, ?, ?, ?, ?, ?, ?)
    """, token_rows)

    print(f"✅ 插入 token 数：{len(token_rows)}")

    # ✅ 填充 word_seq 和 align_ID（只改这里）
    print("🔢 正在填充 word_seq / align_id...")

    cursor.execute("""
        SELECT id
        FROM tokens
        WHERE book_id = ?
          AND chapter = ?
          AND type = 'word'
        ORDER BY chapter, verse, token_id
    """, (book_id, chapter))

    word_tokens = cursor.fetchall()

    for seq, (token_id,) in enumerate(word_tokens, start=1):
        align_id = f"{book_id:02d}_{book_abbr}_{chapter:03d}__{seq:04d}"

        cursor.execute("""
            UPDATE tokens
            SET word_seq = ?,
                align_id = ?
            WHERE id = ?
        """, (seq, align_id, token_id))

    print(f"✅ 已填充 word_seq / align_id 数量：{len(word_tokens)}")

    conn.commit()
    conn.close()

    print("🎉 本章 token 构建完成")


if __name__ == "__main__":
    build_tokens_for_chapter(
        book_abbr="Gen",
        book_id=1,
        chapter=6
    )

📖 处理章节：Gen 6
  需要处理的 verse 数：22
✅ 插入 token 数：1139
🔢 正在填充 word_seq / align_id...
✅ 已填充 word_seq / align_id 数量：538
🎉 本章 token 构建完成


In [ ]:
# mp3 -> wav 
# 增量更新，要改成指定章，为了工作流

import os
import subprocess
import time

MP3_DIR = "outputs/mp3"
WAV_DIR = "outputs/wav"

def batch_mp3_to_wav(mp3_dir: str, wav_dir: str, sample_rate: str = "16000"):
    if not os.path.isdir(mp3_dir):
        raise FileNotFoundError(f"❌ 输入目录不存在：{mp3_dir}")

    os.makedirs(wav_dir, exist_ok=True)

    # 以文件名（不含后缀）做差集
    mp3_files = {
        f[:-4] for f in os.listdir(mp3_dir)
        if f.lower().endswith(".mp3")
        and os.path.isfile(os.path.join(mp3_dir, f))
    }
    wav_files = {
        f[:-4] for f in os.listdir(wav_dir)
        if f.lower().endswith(".wav")
        and os.path.isfile(os.path.join(wav_dir, f))
    }

    to_convert = sorted(mp3_files - wav_files)
    skipped = len(mp3_files & wav_files)

    if skipped:
        print(f"⏭️ 已存在跳过：{skipped} 个")
    if not to_convert:
        print("✅ 全部已同步，无需转换")
        return

    print(f"🎬 待转换 {len(to_convert)} 个\n")

    total = len(to_convert)
    t_start = time.time()

    for i, name in enumerate(to_convert, 1):
        mp3_path = os.path.join(mp3_dir, f"{name}.mp3")
        wav_path = os.path.join(wav_dir, f"{name}.wav")

        cmd = [
            "ffmpeg", "-y",
            "-loglevel", "error",
            "-i", mp3_path,
            "-ar", sample_rate,
            "-ac", "1",
            wav_path
        ]

        s = time.time()
        subprocess.run(cmd, check=True)
        cost = time.time() - s

        print(f"[{i}/{total}] ✅ {name}.wav  ({cost:.2f}s)")

    print(f"\n🏁 完成！共转换 {total} 个，总耗时 {time.time() - t_start:.2f}s")


if __name__ == "__main__":
    batch_mp3_to_wav(MP3_DIR, WAV_DIR, sample_rate="16000")

In [ ]:
# 使用 plaintext 和 wav，输出对应章的 TextGrid
# 增量更新，要改成指定章，为了工作流

import os
import shutil
import subprocess

# ===== 目录 =====
WAV_DIR = "outputs/wav"
TXT_DIR = "outputs/plaintext"
CORPUS_DIR = "corpus"
ALIGN_OUT = "outputs/forcealign"

# ===== MFA 模型路径（✅ 关键修复）=====
dict_path = os.path.expanduser(
    "~/Documents/MFA/pretrained_models/dictionary/english_us_arpa.dict"
)
acoustic_path = os.path.expanduser(
    "~/Documents/MFA/pretrained_models/acoustic/english_us_arpa.zip"
)

# ===== 注入 aligner 环境 =====
conda_prefix = subprocess.check_output(
    ["conda", "info", "--base"], text=True
).strip()
aligner_bin = os.path.join(conda_prefix, "envs", "aligner", "bin")
os.environ["PATH"] = aligner_bin + ":" + os.environ["PATH"]

def prepare_and_align():
    os.makedirs(CORPUS_DIR, exist_ok=True)
    os.makedirs(ALIGN_OUT, exist_ok=True)

    # 已对齐结果
    aligned = {
        f[:-9] for f in os.listdir(ALIGN_OUT)
        if f.endswith(".TextGrid")
    }

    wavs = {f[:-4] for f in os.listdir(WAV_DIR) if f.endswith(".wav")}
    txts = {f[:-4] for f in os.listdir(TXT_DIR) if f.endswith(".txt")}

    ready = wavs & txts
    to_align = sorted(ready - aligned)

    if not to_align:
        print("✅ 所有样本已完成强制对齐")
        return

    print(f"🎯 待对齐：{len(to_align)} 个")

    for name in to_align:
        shutil.copy(
            os.path.join(WAV_DIR, f"{name}.wav"),
            os.path.join(CORPUS_DIR, f"{name}.wav")
        )
        shutil.copy(
            os.path.join(TXT_DIR, f"{name}.txt"),
            os.path.join(CORPUS_DIR, f"{name}.txt")
        )

    cmd = [
        "mfa", "align", CORPUS_DIR,
        dict_path,
        acoustic_path,
        ALIGN_OUT,
        "--clean", "--overwrite"
    ]

    print("🚀 开始强制对齐...\n")
    subprocess.run(cmd, check=True)
    print("\n🏁 强制对齐完成")

if __name__ == "__main__":
    prepare_and_align()

In [4]:
# 使用 TextGrid ，指定章 填充 timestamps 表
# 修复brother's问题

import re
import sqlite3
import os

DB_PATH = "db/bible.db"
TEXTGRID_DIR = "outputs/forcealign"

# ========= 交互输入 =========
book_abbr = input("请输入书卷简称（如 01_Gen）：").strip()
chapter = int(input("请输入章数（如 1）：").strip())

# =========================

filename = f"{book_abbr}_{chapter:03d}_en.TextGrid"
filepath = os.path.join(TEXTGRID_DIR, filename)

if not os.path.exists(filepath):
    raise FileNotFoundError(f"❌ 找不到文件: {filepath}")

with open(filepath, "r", encoding="utf-8") as f:
    content = f.read()


# ======================
# 1️⃣ 只提取 item [1]
# ======================
item1_pattern = re.compile(
    r"item\s*\[1\]:(.*?)(?=\n\s*item\s*\[\d+\]:|\Z)",
    re.DOTALL
)

m = item1_pattern.search(content)
if not m:
    raise ValueError("❌ 未找到 item [1]，请检查 TextGrid 结构")

item1_content = m.group(1)


# ======================
# 2️⃣ 解析 intervals
# ======================
interval_pattern = re.compile(
    r"intervals\s*\[\d+\]:\s*\n"
    r"\s*xmin\s*=\s*([\d.]+)\s*\n"
    r"\s*xmax\s*=\s*([\d.]+)\s*\n"
    r'\s*text\s*=\s*"([^"]*)"',
    re.MULTILINE
)

matches = interval_pattern.findall(item1_content)


# ======================
# 3️⃣ 拆词逻辑（核心）
# ======================
def split_word(word: str):
    """
    brother's -> ["brother", "s"]
    其他 -> [word]
    """
    if word.endswith("'s"):
        return [word[:-2], "s"]
    return [word]


# ======================
# 4️⃣ 写入数据库
# ======================
conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

cursor.execute(
    "SELECT name FROM sqlite_master WHERE type='table' AND name='timestamps'"
)
if cursor.fetchone() is None:
    raise RuntimeError("❌ timestamps 表不存在，请先建表")

word_seq = 0

for xmin, xmax, text in matches:
    text = text.strip()
    if text == "":
        continue

    words = split_word(text)
    duration = float(xmax) - float(xmin)

    # 时间均摊（你之前反对均分，这里先给基础版，下面我会给你“短 s”版本）
    per_word_duration = duration / len(words)

    for i, w in enumerate(words):
        word_seq += 1

        start = float(xmin) + i * per_word_duration
        end = start + per_word_duration

        start_ms = int(round(start * 1000, 0))
        end_ms = int(round(end * 1000, 0))

        align_id = f"{book_abbr}_{chapter:03d}__{word_seq:04d}"

        cursor.execute(
            """
            INSERT INTO timestamps (
                id, chapter, word_seq, word, start_time, end_time
            ) VALUES (?, ?, ?, ?, ?, ?)
            """,
            (align_id, chapter, word_seq, w, start_ms, end_ms)
        )

        print(f"{align_id:20s} | {w:10s} | {start_ms:>6} | {end_ms:>6}")

conn.commit()
conn.close()

print(f"\n✅ 共插入 {word_seq} 条记录（已处理 's 拆分）")

请输入书卷简称（如 01_Gen）：01_Gen
请输入章数（如 1）：6
01_Gen_006__0001     | when       |    120 |    310
01_Gen_006__0002     | people     |    310 |    650
01_Gen_006__0003     | began      |    650 |   1090
01_Gen_006__0004     | to         |   1090 |   1160
01_Gen_006__0005     | multiply   |   1160 |   1860
01_Gen_006__0006     | on         |   1860 |   1980
01_Gen_006__0007     | the        |   1980 |   2050
01_Gen_006__0008     | face       |   2050 |   2390
01_Gen_006__0009     | of         |   2390 |   2510
01_Gen_006__0010     | the        |   2510 |   2600
01_Gen_006__0011     | ground     |   2600 |   3120
01_Gen_006__0012     | and        |   3350 |   3510
01_Gen_006__0013     | daughters  |   3510 |   3920
01_Gen_006__0014     | were       |   3920 |   4040
01_Gen_006__0015     | born       |   4040 |   4390
01_Gen_006__0016     | to         |   4390 |   4500
01_Gen_006__0017     | them       |   4500 |   4840
01_Gen_006__0018     | the        |   5330 |   5420
01_Gen_006__0019     | son

In [11]:
import sqlite3

db_path = "db/bible.db"

conn = sqlite3.connect(db_path)
cursor = conn.cursor()

query = """
SELECT t.id, t.word, tk.token
FROM timestamps t
JOIN tokens tk ON t.id = tk.align_id
"""

cursor.execute(query)

for row in cursor.fetchall():
    id_name = row[0]
    word = (row[1] or "").strip().lower()
    token = (row[2] or "").strip().lower()

    if word != token:
        print(f"发现不一致，id 为: {id_name}")
        print(f"timestamps.word = {row[1]}")
        print(f"tokens.token    = {row[2]}")
        break
else:
    print("✅ 所有 id 对应的 word 和 token 完全一致")

conn.close()

发现不一致，id 为: 12_2K_004__0369
timestamps.word = own
tokens.token    = He


In [31]:
# 检查 align_id是否匹配（播放产生高亮漂移）

import sqlite3

db_path = "db/bible.db"

# ✅ 设置你要检查的 id 前缀
id_prefix = input("前缀").strip()

conn = sqlite3.connect(db_path)
cursor = conn.cursor()

query = """
SELECT t.id, t.word, tk.token
FROM timestamps t
JOIN tokens tk ON t.id = tk.align_id
WHERE t.id LIKE ?
"""

cursor.execute(query, (f"{id_prefix}%",))

for row in cursor.fetchall():
    id_name = row[0]
    word = (row[1] or "").strip().lower()
    token = (row[2] or "").strip().lower()

    if word != token:
        print(f"发现不一致，id 为: {id_name}")
        print(f"timestamps.word = {row[1]}")
        print(f"tokens.token    = {row[2]}")
        break
else:
    print(f"✅ 所有以 '{id_prefix}' 开头的 id，其 word 与 token 完全一致")

conn.close()

前缀01_Gen_004
发现不一致，id 为: 01_Gen_004__0205
timestamps.word = brother's
tokens.token    = brother


In [6]:
# 自动填充 Genesis 7–50 章的 tokens 表

import sqlite3
from typing import List, Tuple

DB_PATH = "db/bible.db"


def segment_english_preserve(text: str) -> List[Tuple[str, str]]:
    if not text:
        return []

    tokens = []
    current = ""
    current_type = None

    def flush():
        nonlocal current, current_type
        if current:
            tokens.append((current, current_type))
            current = ""
            current_type = None

    for ch in text:
        if ch.isalpha():
            t = "word"
        elif ch.isdigit():
            t = "number"
        elif ch.isspace():
            t = "space"
        else:
            t = "punct"

        if t != current_type:
            flush()
            current_type = t

        current += ch

    flush()
    return tokens


def build_tokens_for_chapter(
    book_abbr: str,
    book_id: int,
    chapter: int
):
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()

    print(f"\n📖 处理章节：{book_abbr} {chapter}")

    cursor.execute("""
        SELECT v.id, v.verse, v.text_en
        FROM verses v
        WHERE v.book_id = ?
          AND v.chapter = ?
          AND v.text_en IS NOT NULL
        ORDER BY v.verse
    """, (book_id, chapter))

    verses = cursor.fetchall()
    print(f"  需要处理的 verse 数：{len(verses)}")

    token_rows = []

    for verse_id, verse_num, text_en in verses:
        tokens = segment_english_preserve(text_en)

        reconstructed = "".join(t for t, _ in tokens)
        if reconstructed != text_en:
            print("❌ 拼接不一致")
            print("verse:", verse_id)
            print("原句:", repr(text_en))
            print("拼接:", repr(reconstructed))
            continue

        for idx, (token, token_type) in enumerate(tokens, start=1):
            token_id = f"{book_abbr}.{chapter}.{verse_num}.{idx}"
            token_rows.append((
                token_id,
                book_id,
                chapter,
                verse_num,
                idx,
                token,
                token_type,
                None
            ))

    cursor.executemany("""
        INSERT INTO tokens (
            id,
            book_id,
            chapter,
            verse,
            token_id,
            token,
            type,
            entity_key
        ) VALUES (?, ?, ?, ?, ?, ?, ?, ?)
    """, token_rows)

    print(f"✅ 插入 token 数：{len(token_rows)}")

    # 填充 word_seq / align_id
    print("🔢 正在填充 word_seq / align_id...")

    cursor.execute("""
        SELECT id
        FROM tokens
        WHERE book_id = ?
          AND chapter = ?
          AND type = 'word'
        ORDER BY chapter, verse, token_id
    """, (book_id, chapter))

    word_tokens = cursor.fetchall()

    for seq, (token_id,) in enumerate(word_tokens, start=1):
        align_id = f"{book_id:02d}_{book_abbr}_{chapter:03d}__{seq:04d}"
        cursor.execute("""
            UPDATE tokens
            SET word_seq = ?,
                align_id = ?
            WHERE id = ?
        """, (seq, align_id, token_id))

    print(f"✅ 已填充 word_seq / align_id 数量：{len(word_tokens)}")

    conn.commit()
    conn.close()
    print("🎉 本章 token 构建完成")


if __name__ == "__main__":
    for ch in range(7, 51):   # 7–50
        build_tokens_for_chapter(
            book_abbr="Gen",
            book_id=1,
            chapter=ch
        )


📖 处理章节：Gen 7
  需要处理的 verse 数：24
✅ 插入 token 数：1137
🔢 正在填充 word_seq / align_id...
✅ 已填充 word_seq / align_id 数量：545
🎉 本章 token 构建完成

📖 处理章节：Gen 8
  需要处理的 verse 数：22
✅ 插入 token 数：1155
🔢 正在填充 word_seq / align_id...
✅ 已填充 word_seq / align_id 数量：551
🎉 本章 token 构建完成

📖 处理章节：Gen 9
  需要处理的 verse 数：29
✅ 插入 token 数：1313
🔢 正在填充 word_seq / align_id...
✅ 已填充 word_seq / align_id 数量：621
🎉 本章 token 构建完成

📖 处理章节：Gen 10
  需要处理的 verse 数：32
✅ 插入 token 数：1011
🔢 正在填充 word_seq / align_id...
✅ 已填充 word_seq / align_id 数量：452
🎉 本章 token 构建完成

📖 处理章节：Gen 11
  需要处理的 verse 数：32
✅ 插入 token 数：1339
🔢 正在填充 word_seq / align_id...
✅ 已填充 word_seq / align_id 数量：636
🎉 本章 token 构建完成

📖 处理章节：Gen 12
  需要处理的 verse 数：20
✅ 插入 token 数：1092
🔢 正在填充 word_seq / align_id...
✅ 已填充 word_seq / align_id 数量：511
🎉 本章 token 构建完成

📖 处理章节：Gen 13
  需要处理的 verse 数：18
✅ 插入 token 数：916
🔢 正在填充 word_seq / align_id...
✅ 已填充 word_seq / align_id 数量：433
🎉 本章 token 构建完成

📖 处理章节：Gen 14
  需要处理的 verse 数：24
✅ 插入 token 数：1244
🔢 正在填充 word_seq / align_id...
✅ 已填充

In [8]:
# 使用 TextGrid ，批量填充 timestamps 表（01_Gen 7–50 章）
# 修复 brother's 问题
# ✅ 每一章 id 末尾四位数从 0001 重新计数

import re
import sqlite3
import os

DB_PATH = "db/bible.db"
TEXTGRID_DIR = "outputs/forcealign"

# ========= 固定参数 =========
BOOK_ABBR = "01_Gen"
CHAPTER_START = 7
CHAPTER_END = 50
# ==========================


def split_word(word: str):
    """
    brother's -> ["brother", "s"]
    其他 -> [word]
    """
    if word.endswith("'s"):
        return [word[:-2], "s"]
    return [word]


def process_chapter(book_abbr: str, chapter: int):
    filename = f"{book_abbr}_{chapter:03d}_en.TextGrid"
    filepath = os.path.join(TEXTGRID_DIR, filename)

    if not os.path.exists(filepath):
        print(f"⚠️ 跳过（文件不存在）: {filepath}")
        return 0

    with open(filepath, "r", encoding="utf-8") as f:
        content = f.read()

    # 1️⃣ 只提取 item [1]
    item1_pattern = re.compile(
        r"item\s*\[1\]:(.*?)(?=\n\s*item\s*\[\d+\]:|\Z)",
        re.DOTALL
    )

    m = item1_pattern.search(content)
    if not m:
        print(f"⚠️ 未找到 item [1]，跳过: {filename}")
        return 0

    item1_content = m.group(1)

    # 2️⃣ 解析 intervals
    interval_pattern = re.compile(
        r"intervals\s*\[\d+\]:\s*\n"
        r"\s*xmin\s*=\s*([\d.]+)\s*\n"
        r"\s*xmax\s*=\s*([\d.]+)\s*\n"
        r'\s*text\s*=\s*"([^"]*)"',
        re.MULTILINE
    )

    matches = interval_pattern.findall(item1_content)

    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()

    cursor.execute(
        "SELECT name FROM sqlite_master WHERE type='table' AND name='timestamps'"
    )
    if cursor.fetchone() is None:
        raise RuntimeError("❌ timestamps 表不存在，请先建表")

    # 全局 word_seq（数据库唯一）
    cursor.execute("SELECT COALESCE(MAX(word_seq), 0) FROM timestamps")
    global_word_seq = cursor.fetchone()[0]

    inserted = 0
    chapter_word_seq = 0  # ✅ 本章内序号（id 用）

    for xmin, xmax, text in matches:
        text = text.strip()
        if text == "":
            continue

        words = split_word(text)
        duration = float(xmax) - float(xmin)
        per_word_duration = duration / len(words)

        for i, w in enumerate(words):
            global_word_seq += 1
            chapter_word_seq += 1

            start = float(xmin) + i * per_word_duration
            end = start + per_word_duration

            start_ms = int(round(start * 1000, 0))
            end_ms = int(round(end * 1000, 0))

            # ✅ 本章从 0001 开始
            align_id = f"{book_abbr}_{chapter:03d}__{chapter_word_seq:04d}"

            cursor.execute(
                """
                INSERT INTO timestamps (
                    id, chapter, word_seq, word, start_time, end_time
                ) VALUES (?, ?, ?, ?, ?, ?)
                """,
                (
                    align_id,
                    chapter,
                    global_word_seq,
                    w,
                    start_ms,
                    end_ms,
                ),
            )

            inserted += 1

    conn.commit()
    conn.close()

    print(f"✅ {book_abbr} {chapter:03d} 完成，插入 {inserted} 条")
    return inserted


if __name__ == "__main__":
    total = 0

    for ch in range(CHAPTER_START, CHAPTER_END + 1):
        total += process_chapter(BOOK_ABBR, ch)

    print(f"\n🎉 全部完成！共插入 {total} 条 timestamps 记录")

✅ 01_Gen 007 完成，插入 545 条
✅ 01_Gen 008 完成，插入 551 条
✅ 01_Gen 009 完成，插入 621 条
✅ 01_Gen 010 完成，插入 452 条
✅ 01_Gen 011 完成，插入 636 条
✅ 01_Gen 012 完成，插入 511 条
✅ 01_Gen 013 完成，插入 433 条
✅ 01_Gen 014 完成，插入 564 条
✅ 01_Gen 015 完成，插入 449 条
✅ 01_Gen 016 完成，插入 406 条
✅ 01_Gen 017 完成，插入 656 条
✅ 01_Gen 018 完成，插入 807 条
✅ 01_Gen 019 完成，插入 1029 条
✅ 01_Gen 020 完成，插入 485 条
✅ 01_Gen 021 完成，插入 768 条
✅ 01_Gen 022 完成，插入 578 条
✅ 01_Gen 023 完成，插入 485 条
✅ 01_Gen 024 完成，插入 1716 条
✅ 01_Gen 025 完成，插入 602 条
✅ 01_Gen 026 完成，插入 819 条
✅ 01_Gen 027 完成，插入 1176 条
✅ 01_Gen 028 完成，插入 610 条
✅ 01_Gen 029 完成，插入 764 条
✅ 01_Gen 030 完成，插入 973 条
✅ 01_Gen 031 完成，插入 1400 条
✅ 01_Gen 032 完成，插入 763 条
✅ 01_Gen 033 完成，插入 476 条
✅ 01_Gen 034 完成，插入 751 条
✅ 01_Gen 035 完成，插入 618 条
✅ 01_Gen 036 完成，插入 744 条
✅ 01_Gen 037 完成，插入 888 条
✅ 01_Gen 038 完成，插入 777 条
✅ 01_Gen 039 完成，插入 598 条
✅ 01_Gen 040 完成，插入 531 条
✅ 01_Gen 041 完成，插入 1306 条
✅ 01_Gen 042 完成，插入 973 条
✅ 01_Gen 043 完成，插入 926 条
✅ 01_Gen 044 完成，插入 883 条
✅ 01_Gen 045 完成，插入 713 条
✅ 01_Gen 046 完成，插入 6